<a href="https://colab.research.google.com/github/rezar362/Portfolio/blob/main/stock-gnn/Stock_gnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Downloads 3 years of S&P 500 price data via yfinance and saves to Google Drive.
# On rerun, loads from Drive instead of re-downloading (persistent storage).

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/stock_gnn'

import os
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f"✅ Drive mounted. Save path: {DRIVE_PATH}")

!pip install yfinance torch torch-geometric pandas numpy scikit-learn plotly -q

import yfinance as yf
import pandas as pd
import numpy as np
import torch
import pickle

print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ GPU: {torch.cuda.is_available()}")

TICKERS = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'BRK-B',
    'JPM', 'UNH', 'JNJ', 'V', 'XOM', 'PG', 'MA', 'HD', 'CVX', 'MRK',
    'ABBV', 'PEP', 'KO', 'AVGO', 'COST', 'TMO', 'MCD', 'ACN', 'CSCO',
    'DHR', 'ABT', 'WMT', 'BAC', 'CRM', 'NFLX', 'LIN', 'TXN', 'ADBE',
    'NEE', 'PM', 'RTX', 'HON', 'QCOM', 'LOW', 'UPS', 'IBM', 'GS',
    'MS', 'CAT', 'SPGI', 'BLK', 'AMGN', 'DE', 'AXP', 'ISRG', 'INTU',
    'AMD', 'GILD', 'ADP', 'MDLZ', 'SYK', 'MMC', 'BKNG', 'ADI', 'REGN',
    'CB', 'CI', 'MO', 'SO', 'DUK', 'TJX', 'CL', 'ITW', 'ZTS', 'CME',
    'BSX', 'BDX', 'EQIX', 'NOC', 'LMT', 'WM', 'GD', 'APD', 'NSC',
    'EMR', 'AON', 'MCO', 'FDX', 'EW', 'HUM', 'MAR', 'IDXX', 'KLAC',
    'MCHP', 'LRCX', 'SNPS', 'CDNS', 'PAYX', 'KMB', 'SHW', 'ECL', 'ROP'
]

PRICE_FILE = f'{DRIVE_PATH}/prices.pkl'

if os.path.exists(PRICE_FILE):
    print("📂 Loading prices from Drive...")
    prices = pd.read_pickle(PRICE_FILE)
else:
    print("⬇️  Downloading 3 years of price data...")
    raw = yf.download(TICKERS, start='2021-01-01', end='2024-01-01', auto_adjust=True)
    prices = raw['Close'].dropna(axis=1, thresh=500)
    prices.to_pickle(PRICE_FILE)

returns = prices.pct_change().dropna()
print(f"✅ Returns shape: {returns.shape}")
print("✅ Block 1 complete")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted. Save path: /content/drive/MyDrive/stock_gnn
✅ PyTorch: 2.10.0+cpu
✅ GPU: False
📂 Loading prices from Drive...
✅ Returns shape: (752, 99)
✅ Block 1 complete


In [8]:
# Builds the correlation graph using Spearman correlation on daily returns.
# Nodes = stocks, edges = pairs with correlation > 0.6.
# Node features: mean return, volatility, Sharpe, skewness, 3-month momentum.
# Target label = 1 if stock beats median 30-day forward return, else 0.

import torch
from torch_geometric.data import Data
import pickle
import os

DRIVE_PATH = '/content/drive/MyDrive/stock_gnn'
GRAPH_FILE = f'{DRIVE_PATH}/graph.pkl'

corr_matrix = returns.corr(method='spearman')
tickers = list(corr_matrix.columns)
n = len(tickers)

THRESHOLD = 0.6
edge_index = [[], []]
edge_weight = []

for i in range(n):
    for j in range(i + 1, n):
        corr = corr_matrix.iloc[i, j]
        if corr > THRESHOLD:
            edge_index[0].extend([i, j])
            edge_index[1].extend([j, i])
            edge_weight.extend([corr, corr])

edge_index = torch.tensor(edge_index, dtype=torch.long)
edge_weight = torch.tensor(edge_weight, dtype=torch.float)
print(f"✅ Edges (corr > {THRESHOLD}): {edge_index.shape[1] // 2}")

features = []
for ticker in tickers:
    r = returns[ticker].dropna()
    mean_r = r.mean()
    vol    = r.std()
    sharpe = mean_r / vol if vol > 0 else 0
    skew   = float(r.skew())
    mom_3m = float(r.tail(63).sum())
    features.append([mean_r, vol, sharpe, skew, mom_3m])

x = torch.tensor(features, dtype=torch.float)
x = (x - x.mean(dim=0)) / (x.std(dim=0) + 1e-8)

forward_returns = prices.pct_change(30).shift(-30).iloc[-1]
forward_returns = forward_returns.reindex(tickers).fillna(0)
median_fwd = forward_returns.median()
y = torch.tensor((forward_returns.values > median_fwd).astype(int), dtype=torch.long)

torch.manual_seed(42)
perm = torch.randperm(n)
train_mask = torch.zeros(n, dtype=torch.bool)
val_mask   = torch.zeros(n, dtype=torch.bool)
test_mask  = torch.zeros(n, dtype=torch.bool)
train_mask[perm[:int(0.6*n)]] = True
val_mask[perm[int(0.6*n):int(0.8*n)]] = True
test_mask[perm[int(0.8*n):]] = True

data = Data(x=x, edge_index=edge_index, edge_attr=edge_weight,
            y=y, train_mask=train_mask, val_mask=val_mask, test_mask=test_mask)

with open(GRAPH_FILE, 'wb') as f:
    pickle.dump({'data': data, 'corr_matrix': corr_matrix, 'tickers': tickers}, f)

print(f"✅ Nodes: {data.num_nodes} | Edges: {data.num_edges // 2}")
print(f"✅ Labels: {y.sum().item()} positive, {(y==0).sum().item()} negative")
print(f"✅ Train/Val/Test: {train_mask.sum()}/{val_mask.sum()}/{test_mask.sum()}")
print("✅ Block 2 complete")

✅ Edges (corr > 0.6): 273
✅ Nodes: 99 | Edges: 273
✅ Labels: 0 positive, 99 negative
✅ Train/Val/Test: 59/20/20
✅ Block 2 complete


In [9]:
# Trains a 3-layer Graph Attention Network (GAT) with 4 attention heads.
# Uses cosine LR schedule over 200 epochs.
# Saves best model (by val accuracy) to Drive. Prints loss/accuracy every 20 epochs.

import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv, BatchNorm
import pickle

DRIVE_PATH   = '/content/drive/MyDrive/stock_gnn'
GRAPH_FILE   = f'{DRIVE_PATH}/graph.pkl'
MODEL_FILE   = f'{DRIVE_PATH}/gat_model.pt'
RESULTS_FILE = f'{DRIVE_PATH}/training_results.pkl'

with open(GRAPH_FILE, 'rb') as f:
    saved = pickle.load(f)
data    = saved['data']
tickers = saved['tickers']
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data    = data.to(device)

class StockGAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout, concat=True)
        self.bn1   = BatchNorm(hidden_channels * heads)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=heads, dropout=dropout, concat=True)
        self.bn2   = BatchNorm(hidden_channels * heads)
        self.conv3 = GATConv(hidden_channels * heads, out_channels, heads=1, dropout=dropout, concat=False)

    def forward(self, x, edge_index, edge_attr=None):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index); x = self.bn1(x); x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index); x = self.bn2(x); x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv3(x, edge_index)
        return x

model     = StockGAT(data.num_node_features, 32, 2, heads=4, dropout=0.3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

def train():
    model.train(); optimizer.zero_grad()
    out  = model(data.x, data.edge_index, data.edge_attr)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward(); optimizer.step()
    return float(loss)

@torch.no_grad()
def evaluate(mask):
    model.eval()
    out  = model(data.x, data.edge_index, data.edge_attr)
    pred = out.argmax(dim=1)
    return int((pred[mask] == data.y[mask]).sum()) / int(mask.sum())

history  = {'loss': [], 'train_acc': [], 'val_acc': []}
best_val = 0.0

for epoch in range(1, 201):
    loss      = train()
    train_acc = evaluate(data.train_mask)
    val_acc   = evaluate(data.val_mask)
    scheduler.step()
    history['loss'].append(loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), MODEL_FILE)
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | Train: {train_acc:.3f} | Val: {val_acc:.3f}")

model.load_state_dict(torch.load(MODEL_FILE))
test_acc = evaluate(data.test_mask)
print(f"\n✅ Test Accuracy: {test_acc:.3f} | Best Val: {best_val:.3f}")

with open(RESULTS_FILE, 'wb') as f:
    pickle.dump({'history': history, 'best_val': best_val, 'test_acc': test_acc, 'tickers': tickers}, f)

print("✅ Block 3 complete")

Epoch  20 | Loss: 0.0617 | Train: 1.000 | Val: 1.000
Epoch  40 | Loss: 0.1197 | Train: 1.000 | Val: 1.000
Epoch  60 | Loss: 0.0446 | Train: 1.000 | Val: 1.000
Epoch  80 | Loss: 0.1194 | Train: 1.000 | Val: 1.000
Epoch 100 | Loss: 0.1204 | Train: 1.000 | Val: 1.000
Epoch 120 | Loss: 0.0802 | Train: 1.000 | Val: 1.000
Epoch 140 | Loss: 0.0550 | Train: 1.000 | Val: 1.000
Epoch 160 | Loss: 0.0796 | Train: 1.000 | Val: 1.000
Epoch 180 | Loss: 0.0838 | Train: 1.000 | Val: 1.000
Epoch 200 | Loss: 0.0451 | Train: 1.000 | Val: 1.000

✅ Test Accuracy: 1.000 | Best Val: 1.000
✅ Block 3 complete


In [10]:
# Original Block 4: uses softmax probabilities for outperformance prediction.
# Scores compressed near 0.5 on small graphs — node sizes look similar.
# Label: "Outperform prob", node size range 14-34.

import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv, BatchNorm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx
import pickle

DRIVE_PATH   = '/content/drive/MyDrive/stock_gnn'
GRAPH_FILE   = f'{DRIVE_PATH}/graph.pkl'
MODEL_FILE   = f'{DRIVE_PATH}/gat_model.pt'
RESULTS_FILE = f'{DRIVE_PATH}/training_results.pkl'

with open(GRAPH_FILE, 'rb') as f:
    saved = pickle.load(f)
data        = saved['data']
corr_matrix = saved['corr_matrix']
tickers     = saved['tickers']

with open(RESULTS_FILE, 'rb') as f:
    results = pickle.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data   = data.to(device)

class StockGAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout, concat=True)
        self.bn1   = BatchNorm(hidden_channels * heads)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=heads, dropout=dropout, concat=True)
        self.bn2   = BatchNorm(hidden_channels * heads)
        self.conv3 = GATConv(hidden_channels * heads, out_channels, heads=1, dropout=dropout, concat=False)

    def forward(self, x, edge_index, edge_attr=None):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index); x = self.bn1(x); x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index); x = self.bn2(x); x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv3(x, edge_index)
        return x

model = StockGAT(data.num_node_features, 32, 2, heads=4).to(device)
model.load_state_dict(torch.load(MODEL_FILE, map_location=device))
model.eval()

with torch.no_grad():
    out   = model(data.x, data.edge_index, data.edge_attr)
    probs = F.softmax(out, dim=1)[:, 1].cpu().numpy()
    preds = out.argmax(dim=1).cpu().numpy()

G = nx.Graph()
for i, ticker in enumerate(tickers):
    G.add_node(i, ticker=ticker, prob=float(probs[i]))

edge_arr = data.edge_index.cpu().numpy()
edge_wt  = data.edge_attr.cpu().numpy()
for k in range(0, len(edge_arr[0]), 2):
    i, j = edge_arr[0][k], edge_arr[1][k]
    G.add_edge(i, j, weight=float(edge_wt[k]))

from networkx.algorithms.community import greedy_modularity_communities
communities   = list(greedy_modularity_communities(G))
community_map = {node: cid for cid, c in enumerate(communities) for node in c}

COLORS = ['#7F77DD','#1D9E75','#D85A30','#378ADD','#BA7517','#D4537E','#639922','#E24B4A','#888780']
pos    = nx.spring_layout(G, seed=42, k=2.5, weight='weight')

edge_traces = []
for u, v, d in G.edges(data=True):
    x0, y0 = pos[u]; x1, y1 = pos[v]; w = d['weight']
    edge_traces.append(go.Scatter(
        x=[x0, x1, None], y=[y0, y1, None], mode='lines',
        line=dict(width=w*2.5, color=f'rgba(130,130,130,{w*0.6:.2f})'),
        hoverinfo='none', showlegend=False
    ))

node_x, node_y, node_text, node_color, node_size = [], [], [], [], []
for i in range(len(tickers)):
    x, y = pos[i]
    node_x.append(x); node_y.append(y)
    node_text.append(
        f"<b>{tickers[i]}</b><br>Outperform prob: {probs[i]:.1%}<br>"
        f"Cluster: {community_map.get(i,0)}<br>"
        f"{'↑ Outperform' if preds[i]==1 else '↓ Underperform'}"
    )
    node_color.append(COLORS[community_map.get(i,0) % len(COLORS)])
    node_size.append(14 + probs[i] * 20)

node_trace = go.Scatter(
    x=node_x, y=node_y, mode='markers+text',
    text=tickers, textposition='top center',
    textfont=dict(size=8, color='#444'),
    hovertext=node_text, hoverinfo='text',
    marker=dict(size=node_size, color=node_color,
                line=dict(width=1.5, color='white'), opacity=0.9),
    showlegend=False
)

fig = go.Figure(
    data=edge_traces + [node_trace],
    layout=go.Layout(
        title=dict(
            text=f'Stock Correlation Network — GAT Predictions<br>'
                 f'<sup>Test Acc: {results["test_acc"]:.1%} | Nodes: {len(tickers)} | Edges: {G.number_of_edges()}</sup>',
            x=0.5),
        hovermode='closest', showlegend=False,
        margin=dict(b=20,l=5,r=5,t=60),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        paper_bgcolor='white', plot_bgcolor='white', height=700
    )
)

fig.write_html(f'{DRIVE_PATH}/network_graph_v1.html')
fig.show()
print("✅ Block 4 original complete")

✅ Block 4 original complete


In [11]:
# Updated Block 4: replaces softmax with min-max normalized raw logits.
# Gives wider score spread so node sizes vary more visibly.
# Added score distribution printout and top/bottom 10 stocks.
# Label changed to "Conviction score", node size range 10-40.

import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv, BatchNorm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx
import pickle
import numpy as np

DRIVE_PATH   = '/content/drive/MyDrive/stock_gnn'
GRAPH_FILE   = f'{DRIVE_PATH}/graph.pkl'
MODEL_FILE   = f'{DRIVE_PATH}/gat_model.pt'
RESULTS_FILE = f'{DRIVE_PATH}/training_results.pkl'

with open(GRAPH_FILE, 'rb') as f:
    saved = pickle.load(f)
data        = saved['data']
corr_matrix = saved['corr_matrix']
tickers     = saved['tickers']

with open(RESULTS_FILE, 'rb') as f:
    results = pickle.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data   = data.to(device)

class StockGAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout, concat=True)
        self.bn1   = BatchNorm(hidden_channels * heads)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=heads, dropout=dropout, concat=True)
        self.bn2   = BatchNorm(hidden_channels * heads)
        self.conv3 = GATConv(hidden_channels * heads, out_channels, heads=1, dropout=dropout, concat=False)

    def forward(self, x, edge_index, edge_attr=None):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index); x = self.bn1(x); x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index); x = self.bn2(x); x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv3(x, edge_index)
        return x

model = StockGAT(data.num_node_features, 32, 2, heads=4).to(device)
model.load_state_dict(torch.load(MODEL_FILE, map_location=device))
model.eval()

with torch.no_grad():
    out   = model(data.x, data.edge_index, data.edge_attr)
    preds = out.argmax(dim=1).cpu().numpy()
    raw_scores = out[:, 1].cpu().numpy()
    probs = (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())

print("── Score Distribution ──")
print(f"   Min   : {probs.min():.3f}")
print(f"   Max   : {probs.max():.3f}")
print(f"   Mean  : {probs.mean():.3f}")
print(f"   Median: {np.median(probs):.3f}")

print(f"\n── Top 10 stocks by score ──")
top10 = np.argsort(probs)[::-1][:10]
for i in top10:
    print(f"   {tickers[i]:6s} | Score: {probs[i]:.3f} | {'↑ Outperform' if preds[i]==1 else '↓ Underperform'}")

print(f"\n── Bottom 10 stocks by score ──")
bot10 = np.argsort(probs)[:10]
for i in bot10:
    print(f"   {tickers[i]:6s} | Score: {probs[i]:.3f} | {'↑ Outperform' if preds[i]==1 else '↓ Underperform'}")

G = nx.Graph()
for i, ticker in enumerate(tickers):
    G.add_node(i, ticker=ticker, prob=float(probs[i]))

edge_arr = data.edge_index.cpu().numpy()
edge_wt  = data.edge_attr.cpu().numpy()
for k in range(0, len(edge_arr[0]), 2):
    i, j = edge_arr[0][k], edge_arr[1][k]
    G.add_edge(i, j, weight=float(edge_wt[k]))

from networkx.algorithms.community import greedy_modularity_communities
communities   = list(greedy_modularity_communities(G))
community_map = {node: cid for cid, c in enumerate(communities) for node in c}

COLORS = ['#7F77DD','#1D9E75','#D85A30','#378ADD','#BA7517','#D4537E','#639922','#E24B4A','#888780']
pos    = nx.spring_layout(G, seed=42, k=2.5, weight='weight')

edge_traces = []
for u, v, d in G.edges(data=True):
    x0, y0 = pos[u]; x1, y1 = pos[v]; w = d['weight']
    edge_traces.append(go.Scatter(
        x=[x0, x1, None], y=[y0, y1, None], mode='lines',
        line=dict(width=w*2.5, color=f'rgba(130,130,130,{w*0.6:.2f})'),
        hoverinfo='none', showlegend=False
    ))

node_x, node_y, node_text, node_color, node_size = [], [], [], [], []
for i in range(len(tickers)):
    x, y = pos[i]
    node_x.append(x); node_y.append(y)
    node_text.append(
        f"<b>{tickers[i]}</b><br>"
        f"Conviction score: {probs[i]:.3f}<br>"
        f"Cluster: {community_map.get(i,0)}<br>"
        f"{'↑ Outperform' if preds[i]==1 else '↓ Underperform'}"
    )
    node_color.append(COLORS[community_map.get(i,0) % len(COLORS)])
    node_size.append(10 + probs[i] * 30)

node_trace = go.Scatter(
    x=node_x, y=node_y, mode='markers+text',
    text=tickers, textposition='top center',
    textfont=dict(size=8, color='#444'),
    hovertext=node_text, hoverinfo='text',
    marker=dict(size=node_size, color=node_color,
                line=dict(width=1.5, color='white'), opacity=0.9),
    showlegend=False
)

fig = go.Figure(
    data=edge_traces + [node_trace],
    layout=go.Layout(
        title=dict(
            text=f'Stock Correlation Network — GAT Predictions<br>'
                 f'<sup>Test Acc: {results["test_acc"]:.1%} | Nodes: {len(tickers)} | Edges: {G.number_of_edges()}</sup>',
            x=0.5),
        hovermode='closest', showlegend=False,
        margin=dict(b=20,l=5,r=5,t=60),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        paper_bgcolor='white', plot_bgcolor='white', height=700
    )
)

fig.write_html(f'{DRIVE_PATH}/network_graph_v2.html')
fig.show()

print(f"\n✅ Communities detected: {len(communities)}")
print("✅ Block 4 updated complete")

── Score Distribution ──
   Min   : 0.000
   Max   : 1.000
   Mean  : 0.639
   Median: 0.648

── Top 10 stocks by score ──
   NFLX   | Score: 1.000 | ↓ Underperform
   NSC    | Score: 0.987 | ↓ Underperform
   ABT    | Score: 0.881 | ↓ Underperform
   TMO    | Score: 0.880 | ↓ Underperform
   DHR    | Score: 0.871 | ↓ Underperform
   EW     | Score: 0.853 | ↓ Underperform
   UPS    | Score: 0.848 | ↓ Underperform
   FDX    | Score: 0.848 | ↓ Underperform
   ZTS    | Score: 0.842 | ↓ Underperform
   SYK    | Score: 0.805 | ↓ Underperform

── Bottom 10 stocks by score ──
   CVX    | Score: 0.000 | ↓ Underperform
   XOM    | Score: 0.000 | ↓ Underperform
   AON    | Score: 0.314 | ↓ Underperform
   ABBV   | Score: 0.335 | ↓ Underperform
   MRK    | Score: 0.403 | ↓ Underperform
   WM     | Score: 0.407 | ↓ Underperform
   MCD    | Score: 0.408 | ↓ Underperform
   LMT    | Score: 0.447 | ↓ Underperform
   NOC    | Score: 0.447 | ↓ Underperform
   GD     | Score: 0.457 | ↓ Underperform



✅ Communities detected: 30
✅ Block 4 updated complete
